# Weekly regime-aware momentum backtest

This notebook reports performance, charts the portfolio over time, and records every position change.

In [ ]:
import sys
from pathlib import Path

candidates = [Path.cwd(), *Path.cwd().parents]
project_root = next((path for path in candidates if (path / 'src' / 'trading').is_dir()), None)
if project_root is None:
    raise RuntimeError('Could not locate the project root containing src/trading')
sys.path.insert(0, str(project_root / 'src'))

from trading.config import load_config
from trading.data import load_prices
from trading.strategy import weekly_momentum_weights
from trading.backtest import plot_backtest, plot_backtest_periods, run_backtest, format_performance_report

config = load_config(project_root / 'config.yaml')
prices = load_prices(project_root / 'data' / 'processed' / 'prices.parquet')
weights = weekly_momentum_weights(prices, config['benchmark'], config['top_n'], config['lookback_days'], config['skip_days'], config['volatility_days'], config['max_position_weight'], config['max_equity_exposure'], config['defensive_exposure'], config['stop_loss_pct'], config['drawdown_start'], config['drawdown_full_defense'])
ledger, holdings, trades = run_backtest(prices, weights, config['initial_capital'], config['commission_bps'], config['half_spread_bps'], config['slippage_bps'], config['market_impact_bps'])
print(format_performance_report(ledger, config['initial_capital']))

In [ ]:
plot_backtest(ledger, trades, prices[config['benchmark']])

In [ ]:
plot_backtest_periods(ledger, prices[config['benchmark']])

In [ ]:
trades.style.format({
    'weight_before': '{:.3%}',
    'weight_after': '{:.3%}',
    'weight_change': '{:.3%}',
    'execution_price': '${:,.3f}',
    'notional': '${:,.3f}',
    'estimated_cost': '${:,.3f}',
    'realized_profit_loss': '${:,.3f}',
    'net_profit_loss': '${:,.3f}',
    'profit_loss_pct': '{:.3%}',
    'cumulative_net_profit_loss': '${:,.3f}',
})